# AMD_ROCm_Qwen3.5_LoRA ⚡️
**目标**: 用尽量少的依赖跑通 Qwen3.5-4B 的 LoRA 微调、保存、加载和可选 vLLM 推理。

功能：
- 使用 `torch.bfloat16 + peft LoRA + gradient checkpointing`。
- 内置一个很小的中文指令数据集，先用于 smoke test。
- 先保存 LoRA adapter，可选合并成完整模型后交给 vLLM。

> 前置：已完成 `AMD_ROCm_Quick_Start.ipynb` 的环境验证，最好熟悉了 `AMD_ROCm_LLM_Inference.ipynb`的大模型推理原理。

In [81]:
## 1)基础环境检查、配置、模型下载
import os
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch

assert torch.cuda.is_available(), "No GPU found by torch.cuda. Please check the ROCm/PyTorch environment."

print("torch:", torch.__version__)
print("hip:", getattr(torch.version, "hip", None))
print("device count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, memory={props.total_memory / 1024**3:.1f} GB")


WORK_DIR = Path("/mnt/workspace/amd/qwen35_4b_lora_demo")
print("work dir:", WORK_DIR)

MODEL_CACHE_DIR = WORK_DIR / "models"
OUTPUT_DIR = WORK_DIR / "outputs"
ADAPTER_DIR = OUTPUT_DIR / "qwen35_4b_lora_adapter"
MERGED_DIR = OUTPUT_DIR / "qwen35_4b_lora_merged"
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from modelscope import snapshot_download
MODEL_ID = "Qwen/Qwen3.5-4B"
model_dir = snapshot_download(
    MODEL_ID,
    cache_dir=str(MODEL_CACHE_DIR),
)
print("model dir:", model_dir)

MAX_LENGTH = 1024
MAX_STEPS = 40
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4

torch: 2.10.0+git8514f05
hip: 7.2.53211
device count: 1
GPU 0: , memory=191.7 GB
work dir: /mnt/workspace/amd/qwen35_4b_lora_demo


2026-05-31 11:33:43,148 - modelscope - INFO - Target directory already exists, skipping creation.


model dir: /mnt/workspace/amd/qwen35_4b_lora_demo/models/Qwen/Qwen3___5-4B


In [82]:
## 2）微调训练数据准备
from datasets import Dataset
from transformers import AutoTokenizer

SYSTEM_PROMPT = "你是一个简洁、准确的中文技术助手。"
raw_examples = [
    {
        "instruction": "用三句话解释什么是 ROCm。",
        "output": "ROCm 是 AMD 的开源 GPU 计算软件栈。它提供编译器、运行时和深度学习库，让 PyTorch 等框架可以在 AMD GPU 上运行。对模型训练和推理来说，ROCm 扮演的角色类似 CUDA 在 NVIDIA GPU 上的角色。",
    },
    {
        "instruction": "给出一个适合新手的 LoRA 微调建议。",
        "output": "先用很小的数据集和较短的 max_steps 跑通流程。确认 loss 能正常下降、adapter 能保存和加载后，再扩大数据量。ROCm 环境下优先使用 bf16 LoRA，避免一开始就引入复杂的量化依赖。",
    },
    {
        "instruction": "请把下面的话改写得更正式：这个模型跑得挺快。",
        "output": "该模型在当前测试环境下展现出较高的推理效率。",
    },
]

train_examples = raw_examples * 40
dataset = Dataset.from_list(train_examples)
print(dataset)
print(dataset[0])


tokenizer = AutoTokenizer.from_pretrained(
    model_dir,
    trust_remote_code=True,
    use_fast=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

def tokenize_example(example):
    return tokenizer(
        example["text"],
        max_length=MAX_LENGTH,
        truncation=True,
    )

formatted = dataset.map(format_example, remove_columns=dataset.column_names)
tokenized = formatted.map(tokenize_example, remove_columns=formatted.column_names)

print(tokenized)
print(tokenizer.decode(tokenized[0]["input_ids"][:200]))

Dataset({
    features: ['instruction', 'output'],
    num_rows: 120
})
{'instruction': '用三句话解释什么是 ROCm。', 'output': 'ROCm 是 AMD 的开源 GPU 计算软件栈。它提供编译器、运行时和深度学习库，让 PyTorch 等框架可以在 AMD GPU 上运行。对模型训练和推理来说，ROCm 扮演的角色类似 CUDA 在 NVIDIA GPU 上的角色。'}


Map: 100%|██████████| 120/120 [00:00<00:00, 4889.61 examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 120
})
<|im_start|>system
你是一个简洁、准确的中文技术助手。<|im_end|>
<|im_start|>user
用三句话解释什么是 ROCm。<|im_end|>
<|im_start|>assistant
<think>

</think>

ROCm 是 AMD 的开源 GPU 计算软件栈。它提供编译器、运行时和深度学习库，让 PyTorch 等框架可以在 AMD GPU 上运行。对模型训练和推理来说，ROCm 扮演的角色类似 CUDA 在 NVIDIA GPU 上的角色。<|im_end|>



In [91]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 426/426 [01:25<00:00,  4.97it/s]


trainable params: 21,233,664 || all params: 4,226,984,960 || trainable%: 0.5023


In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "trainer"),
    max_steps=MAX_STEPS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    bf16=True,
    fp16=False,
    logging_steps=1,
    save_steps=20,
    save_total_limit=2,
    optim="adamw_torch",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

trainer.train()

## 默认只保存lora适配器
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("saved adapter:", ADAPTER_DIR)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
1,2.632481
2,2.510579
3,2.315396
4,1.549927
5,1.061697
6,0.664221
7,0.396931
8,0.179193
9,0.077527
10,0.115420


saved adapter: /mnt/workspace/amd/qwen35_4b_lora_demo/outputs/qwen35_4b_lora_adapter


In [ ]:
# Optional: merge LoRA into a standalone model directory for vLLM.
merged_model = model.merge_and_unload()
merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
    max_shard_size="4GB",
)
tokenizer.save_pretrained(MERGED_DIR)
print("saved merged model:", MERGED_DIR)

saved merged model: /mnt/workspace/amd/qwen35_4b_lora_demo/outputs/qwen35_4b_lora_merged


In [114]:
# Optional: inference on the merged model.
device = torch.device("cuda")
model = AutoModelForCausalLM.from_pretrained(
    str(MERGED_DIR),
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(str(MERGED_DIR))

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "给出三个在 AMD ROCm 上微调大模型的实践建议。(不要输出think过程)"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=512)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights: 100%|██████████| 426/426 [00:03<00:00, 133.88it/s]
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


system
你是一个简洁、准确的中文技术助手。
user
给出三个在 AMD ROCm 上微调大模型的实践建议。(不要输出think过程)
assistant
<think>
用户需要三个在 ROCm 上微调大模型的实践建议。我应该给出简洁、可操作的建议。

1. 先用较小的数据集和较短的 max_steps 验证流程。
2. 使用 LoRA 适配器而不是从头训练 ROCm ROCm。
3. 确认 bf16 LoRA 模型可以正常保存和加载。

这样回答比较合适。
</think>

先用很小的数据集和较短的 max_steps 跑通流程。确认 ROCm 环境下 LoRA 可以正常保存和加载后，再扩大数据量。优先使用 bf16 LoRA 模型，避免一开始就引入复杂的量化依赖。



In [115]:
# Optional: inference on the isolated model.
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)
tokenizer = AutoTokenizer.from_pretrained(model_dir) #lora不改变词典，所以从原模型或者adapter目录导入均可

with torch.no_grad():
    print(f"\n{'='*20}" + " 原模型预测 " + f"{'='*20}")
    outputs = model.generate(**inputs, max_new_tokens=512)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print(f"\n{'='*60}")

adapter = PeftModel.from_pretrained(model, ADAPTER_DIR)

with torch.no_grad():
    print(f"\n{'='*20}" + " Lora模型预测 " + f"{'='*20}")
    outputs = adapter.generate(**inputs, max_new_tokens=512)
    print(f"\n{'='*60}")
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights: 100%|██████████| 426/426 [00:03<00:00, 137.14it/s]
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



==================== 原模型预测 ====================
system
你是一个简洁、准确的中文技术助手。
user
给出三个在 AMD ROCm 上微调大模型的实践建议。(不要输出think过程)
assistant
<think>
Thinking Process:

1.  **Analyze the Request:**
    *   Role: Concise and accurate Chinese technical assistant.
    *   Task: Provide three practical suggestions for fine-tuning large models on AMD ROCm.
    *   Constraint: Do not output the thinking process (no "think" content).
    *   Language: Chinese.

2.  **Identify Key Challenges with ROCm:**
    *   ROCm (Radeon Open Compute) is AMD's GPU computing platform.
    *   Compared to NVIDIA CUDA, it has a steeper learning curve and fewer ecosystem tools.
    *   Fine-tuning large models (LLMs) on ROCm often involves specific libraries (like `llama.cpp`, `Hugging Face Transformers` with ROCm support, `DeepSpeed`, `vLLM` with ROCm support).
    *   Common issues: Memory management, driver compatibility, compilation issues, lack of optimization compared to CUDA.

3.  **Drafting Suggestions:**
    *   

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



==================== Lora模型预测 ====================

system
你是一个简洁、准确的中文技术助手。
user
给出三个在 AMD ROCm 上微调大模型的实践建议。(不要输出think过程)
assistant
<think>
用户需要三个在 ROCm 上微调大模型的实践建议。我应该给出简洁、可操作的建议。

1. 先用较小的数据集和较短的 max_steps 验证流程。
2. 使用 LoRA 适配器而不是从头训练 ROCm 模型。
3. 确认 bf16 LoRA 可以正常保存和加载。

这样用户可以先确认环境没问题，再扩大数据量。
</think>

先用很小的数据集和较短的 max_steps 跑通流程。确认 ROCm 环境下 LoRA 可以正常保存和加载后，再扩大数据量。优先使用 bf16 LoRA 适配器，避免一开始就引入复杂的量化依赖。



## 调参建议

在 192 GB 显存的 AMD GPU 上，Qwen3.5-4B 的 bf16 LoRA 应该比较宽松。建议先保持 `MAX_STEPS=40` 跑通流程，再逐步调整：

- 把 `MAX_STEPS` 提高到 `500` 或更多，用真实数据训练。
- 如果显存稳定，可以把 `PER_DEVICE_BATCH_SIZE` 从 `2` 提到 `4` 或 `8`。
- 如果遇到显存压力，先降低 `MAX_LENGTH`，再降低 batch size。
- ROCm 上先确认 bf16 LoRA 稳定，再考虑更复杂的量化方案。